<a href="https://colab.research.google.com/github/1kaiser/opyf_colab/blob/main/jax_3d_reconstruction_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# JAX 3D Reconstruction Pipeline (Zonal)
🚀 This notebook provides a high-performance JAX ecosystem for metric 3D reconstruction.

All code and datasets are now integrated directly into the **opyf_colab** repository.

## 1. Environment Setup

In [1]:
# @title 1.1 Setup & Dependencies
import sys
import os

def is_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

IN_COLAB = is_colab()

if IN_COLAB:
    print("Running in Google Colab")
    os.system("pip install -q jax[tpu] -f https://storage.googleapis.com/jax-releases/libtpu_releases.html")
    os.system("pip install -q flax open3d tqdm opencv-python")

    if not os.path.exists("/content/opyf_colab"):
        os.system("git clone https://github.com/1kaiser/opyf_colab.git /content/opyf_colab")

    sys.path.append("/content/opyf_colab")
    sys.path.append("/content/opyf_colab/models/jax")
    os.chdir("/content/opyf_colab")
else:
    print("Running Locally")
    project_root = os.getcwd()
    if project_root not in sys.path:
        sys.path.append(project_root)
    models_path = os.path.join(project_root, "models/jax")
    if models_path not in sys.path:
        sys.path.append(models_path)

import jax
import flax
import open3d as o3d
from tqdm import tqdm
import cv2
print(f"JAX Devices: {jax.devices()}")


Running in Google Colab
JAX Devices: [CudaDevice(id=0)]


In [ ]:
!wget -O /content/opyf_colab/nerf_real_360.zip https://huggingface.co/datasets/1kaiser/NERF_360/resolve/main/nerf_real_360.zip
!unzip -n nerf_real_360.zip

--2026-03-11 04:27:42--  https://huggingface.co/datasets/1kaiser/NERF_360/resolve/main/nerf_real_360.zip
Resolving huggingface.co (huggingface.co)... 18.239.69.71, 18.239.69.31, 18.239.69.83, ...
Connecting to huggingface.co (huggingface.co)|18.239.69.71|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/69b0eeabff671955d30d1436/4e5173626a10d70f2717396b8035c9d0ecdc50b20a5c3b3afe6fa1c36aed79c1?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27nerf_real_360.zip%3B+filename%3D%22nerf_real_360.zip%22%3B&response-content-type=application%2Fzip&Expires=1773206862&Policy=eyJTdGF0ZW1lbnQiOlt7IkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiRXBvY2hUaW1lIjoxNzczMjA2ODYyfX0sIlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjliMGVlYWJmZjY3MTk1NWQzMGQxNDM2LzRlNTE3MzYyNmExMGQ3MGYyNzE3Mzk2YjgwMzVjOWQwZWNkYzUwYjIwYTVjM2IzYWZlNmZhMWMzNmFlZDc5YzFcXD9yZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPSomcmVzcG9uc2UtY29udGVudC10eX

In [3]:
# !wget -O /content/opyf_colab/weights/depth_pro.msgpack https://github.com/1kaiser/opyf_colab/releases/download/v1.0.0/depth_pro.msgpack
# !wget -O /content/opyf_colab/weights/superpoint.msgpack https://github.com/1kaiser/opyf_colab/releases/download/v1.0.0/superpoint.msgpack
# !wget -O /content/opyf_colab/weights/superpoint_lightglue.msgpack https://github.com/1kaiser/opyf_colab/releases/download/v1.0.0/superpoint_lightglue.msgpack



/content/opyf_colab/weights/depth_pro.msgpack: No such file or directory
/content/opyf_colab/weights/superpoint.msgpack: No such file or directory
/content/opyf_colab/weights/superpoint_lightglue.msgpack: No such file or directory


In [4]:
# @title 1.2 Weights Management
import os

def download_weights():
    os.makedirs("weights", exist_ok=True)
    BASE_URL = "https://github.com/1kaiser/opyf_colab/releases/download/v1.0.0"

    print("Downloading/Verifying weights...")

    models = {
        "depth_pro.msgpack": None,
        "superpoint.msgpack": None,
        "superpoint_lightglue.msgpack": None
    }

    for target, parts in models.items():
        target_path = os.path.join("weights", target)
        if os.path.exists(target_path):
            print(f"✅ {target} already exists.")
            continue

        print(f"Downloading {target}...")
        url = f"{BASE_URL}/{target}"
        if IN_COLAB:
            os.system(f"wget -q --show-progress {url} -O {target_path}")
        else:
            import subprocess
            subprocess.run(["wget", "-q", "--show-progress", url, "-O", target_path])

    print("\n✅ Required weights ready.")

if IN_COLAB:
    download_weights()
else:
    required = ["depth_pro.msgpack", "superpoint.msgpack", "superpoint_lightglue.msgpack"]
    missing = [r for r in required if not os.path.exists(os.path.join("weights", r))]
    if missing:
        print(f"Missing weights: {missing}. Downloading...")
        download_weights()
    else:
        print("✅ Local weights detected.")


Downloading/Verifying weights...

✅ Required weights ready.


## 2. Standalone Model Testing

In [5]:
!ls -h /content/opyf_colab/pinecone/images

ls: cannot access '/content/opyf_colab/pinecone/images': No such file or directory


In [6]:
if IN_COLAB:   test_img1 = "/content/opyf_colab/pinecone/images/IMG_7238.JPG"
else: test_img1 = "./content/opyf_colab/pinecone/images/IMG_7238.JPG"
os.makedirs("output", exist_ok=True)
os.system(f"python -m inference.infer_depth_pro --image {test_img1} --weights weights/depth_pro.msgpack --output output/depth_result.jpg")

256

## 3. Zonal 3D Reconstruction

In [7]:
from pipelines.pipeline_jax import ReconstructionPipeline

pipeline = ReconstructionPipeline("weights/depth_pro.msgpack", "weights/superpoint.msgpack", "weights/superpoint_lightglue.msgpack")

T_globals = pipeline.run("/content/opyf_colab/pinecone/images", output_path="output/pinecone_reconstruction", num_zones=3, radial_clip=0.7)

Loading weights from weights/depth_pro.msgpack, weights/superpoint.msgpack, weights/superpoint_lightglue.msgpack...
Loading Depth Pro weights...


ValueError: Unpack failed: incomplete input

## 4. Visualization

In [ ]:
#################live glb viewer#################
from IPython.display import HTML
import base64

# Load GLB as base64
with open("/content/opyf_colab/output/pinecone_reconstruction/mesh_clip0.7.glb", "rb") as f:
    glb_data = base64.b64encode(f.read()).decode("utf-8")

HTML(f"""
<iframe srcdoc="
<!DOCTYPE html>
<html lang='en'>
  <head>
    <script type='module' src='https://unpkg.com/@google/model-viewer/dist/model-viewer.min.js'></script>
    <style>html, body {{ margin: 0; height: 100%; }}</style>
  </head>
  <body>
    <model-viewer
      src='data:model/gltf-binary;base64,{glb_data}'
      alt='3D scene'
      auto-rotate
      camera-controls
      background-color='#FFFFFF'
      style='width:100%; height:100%;'>
    </model-viewer>
  </body>
</html>
" width="100%" height="600px" style="border:0;"></iframe>
""")